In [ ]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HF_TOKEN", "")
MODEL_ID = "facebook/convnext-base-224"

IMAGE_PATH = r"google_test_images\adversarial\img_0000.png"


client = InferenceClient(
    provider="hf-inference",
    api_key=HF_TOKEN
)

result = client.image_classification(
    IMAGE_PATH,
    model=MODEL_ID
)

print("\nPredictions:\n")

for i, pred in enumerate(result[:10], start=1):
    print(
        f"{i:2d}. "
        f"{pred.label:40s} "
        f"{pred.score * 100:.2f}%"
    )

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from huggingface_hub import InferenceClient


# ============================================================
# CONFIG
# ============================================================

# MODEL_ID = "google/vit-base-patch16-224"

# IMAGE_FOLDER = r"google_test_images\adversarial"
# OUTPUT_CSV = "hf_vit_adversarial_results.csv"

# IMAGE_FOLDER = r"google_test_images\benign"
# OUTPUT_CSV = "hf_vit_benign_results.csv"

MODEL_ID = "facebook/convnext-base-224"

IMAGE_FOLDER = r"google_test_images\SMER_ViT"
OUTPUT_CSV = "hf_convnext_adversarial_results_SMER_ViT.csv" 

# IMAGE_FOLDER = r"google_test_images\benign"
# OUTPUT_CSV = "hf_convnext_benign_results.csv"


TOP_K = 10


# ============================================================
# CLIENT
# ============================================================

client = InferenceClient(
    provider="hf-inference",
    api_key=HF_TOKEN
)


# ============================================================
# EXACT SAME 75 IMAGES USED WITH IMAGGA / AWS
# ============================================================

# reference = pd.read_csv(
#     "imagga_adversarial_results.csv"
# )

# images = reference["image"].tolist()

extensions = (
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
)

images = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(extensions)
])

print("=" * 60)
print("HUGGING FACE ViT TEST")
print("=" * 60)
print("Model:", MODEL_ID)
print("Folder:", IMAGE_FOLDER)
print("Number of images:", len(images))
print("=" * 60)


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_image(image_path):

    result = client.image_classification(
        image_path,
        model=MODEL_ID
    )

    return result


# ============================================================
# PROCESS
# ============================================================

rows = []

for filename in tqdm(images):

    image_path = os.path.join(
        IMAGE_FOLDER,
        filename
    )

    row = {
        "image": filename
    }

    try:

        predictions = predict_image(
            image_path
        )

        row["status"] = "success"

        for i, pred in enumerate(
            predictions[:TOP_K],
            start=1
        ):

            row[f"top{i}_label"] = pred.label

            row[f"top{i}_confidence"] = (
                pred.score * 100
            )

    except Exception as e:

        row["status"] = "failed"
        row["error"] = str(e)

    rows.append(row)


# ============================================================
# SAVE
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nFinished.")

print(
    "Successful:",
    (df["status"] == "success").sum()
)

print(
    "Failed:",
    (df["status"] == "failed").sum()
)

print(
    "Saved:",
    OUTPUT_CSV
)

In [ ]:
import pandas as pd
import re


# ============================================================
# FILES
# ============================================================

LABELS_CSV = "google_test_images/labels.csv"

# BENIGN_CSV = "hf_vit_benign_results.csv"
# ADV_CSV = "hf_vit_adversarial_results.csv"

BENIGN_CSV = "hf_convnext_benign_results.csv"
ADV_CSV = "hf_convnext_adversarial_results_SMER_ViT.csv"



OUTPUT_CSV = "hf_convnext_evaluation_detailed.csv"


# ============================================================
# LOAD
# ============================================================

labels = pd.read_csv(LABELS_CSV)
benign = pd.read_csv(BENIGN_CSV)
adv = pd.read_csv(ADV_CSV)

if "status" in benign.columns:
    benign = benign[benign["status"] == "success"].copy()

if "status" in adv.columns:
    adv = adv[adv["status"] == "success"].copy()


# ============================================================
# MERGE
# ============================================================

df = labels.merge(
    benign,
    left_on="benign_image",
    right_on="image",
    how="inner"
)

df = df.merge(
    adv,
    left_on="adversarial_image",
    right_on="image",
    how="inner",
    suffixes=("_benign", "_adv")
)

print("Number of paired images:", len(df))


# ============================================================
# NORMALIZATION
# ============================================================

def normalize(text):

    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = text.replace("-", " ")
    text = text.replace("_", " ")

    text = re.sub(r"[^a-z0-9, ]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text


# ============================================================
# IMAGENET LABEL MATCH
# ============================================================
#
# HF labels can look like:
#
# "hammerhead, hammerhead shark"
#
# while GT may be:
#
# "hammerhead"
#
# We split aliases by comma and check each one.
# ============================================================

def imagenet_match(gt, prediction):

    gt = normalize(gt)
    prediction = normalize(prediction)

    if not gt or not prediction:
        return False

    aliases = [
        x.strip()
        for x in prediction.split(",")
    ]

    for alias in aliases:

        if alias == gt:
            return True

        # useful for cases such as:
        # hammerhead <-> hammerhead shark
        if gt in alias or alias in gt:
            return True

    return False


# ============================================================
# CORRECT @ K
# ============================================================

def correct_at_k(row, suffix, k):

    gt = row["label_name"]

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col not in row.index:
            continue

        if imagenet_match(
            gt,
            row[col]
        ):
            return True

    return False


# ============================================================
# ACCURACY
# ============================================================

for k in [1, 5, 10]:

    df[f"benign_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "benign",
            k
        ),
        axis=1
    )

    df[f"adv_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "adv",
            k
        ),
        axis=1
    )


print("\n============================================")
print("          HF ViT ROBUSTNESS RESULTS")
print("============================================")


for k in [1, 5, 10]:

    benign_acc = (
        df[f"benign_top{k}_correct"]
        .mean() * 100
    )

    adv_acc = (
        df[f"adv_top{k}_correct"]
        .mean() * 100
    )

    drop = benign_acc - adv_acc

    print(f"\nTOP-{k}")

    print(
        f"Benign accuracy:       "
        f"{benign_acc:.2f}%"
    )

    print(
        f"Adversarial accuracy:  "
        f"{adv_acc:.2f}%"
    )

    print(
        f"Accuracy drop:         "
        f"{drop:.2f} pp"
    )


# ============================================================
# ASR
# ============================================================

print("\n============================================")
print("             ATTACK SUCCESS RATE")
print("============================================")


for k in [1, 5, 10]:

    benign_correct = df[
        df[f"benign_top{k}_correct"]
    ]

    successful = benign_correct[
        ~benign_correct[
            f"adv_top{k}_correct"
        ]
    ]

    if len(benign_correct) > 0:

        asr = (
            len(successful)
            /
            len(benign_correct)
            * 100
        )

    else:
        asr = 0


    print(f"\nTop-{k}:")

    print(
        "Benign correct:",
        len(benign_correct)
    )

    print(
        "Successfully attacked:",
        len(successful)
    )

    print(
        f"ASR: {asr:.2f}%"
    )


# ============================================================
# TOP-1 PREDICTION CHANGE
# ============================================================

df["top1_changed"] = (

    df["top1_label_benign"]
    .fillna("")
    .apply(normalize)

    !=

    df["top1_label_adv"]
    .fillna("")
    .apply(normalize)
)

change_rate = (
    df["top1_changed"].mean()
    * 100
)


# ============================================================
# CONFIDENCE DROP
# ============================================================

df["top1_confidence_drop"] = (
    df["top1_confidence_benign"]
    -
    df["top1_confidence_adv"]
)

mean_conf_drop = (
    df["top1_confidence_drop"]
    .mean()
)


# ============================================================
# TOP-5 RETENTION
# ============================================================

def get_topk(row, suffix, k):

    values = []

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col in row.index and pd.notna(row[col]):

            values.append(
                normalize(row[col])
            )

    return set(values)


def retention(row):

    b = get_topk(
        row,
        "benign",
        5
    )

    a = get_topk(
        row,
        "adv",
        5
    )

    if len(b) == 0:
        return 0

    return len(
        b.intersection(a)
    ) / len(b)


df["top5_retention"] = df.apply(
    retention,
    axis=1
)

mean_retention = (
    df["top5_retention"].mean()
    * 100
)


df["top5_complete_disruption"] = (
    df["top5_retention"] == 0
)

complete_disruption = (
    df[
        "top5_complete_disruption"
    ].mean()
    * 100
)


# ============================================================
# PRINT EXTRA METRICS
# ============================================================

print("\n============================================")
print("          PREDICTION DISRUPTION")
print("============================================")

print(
    f"Top-1 Label Change Rate: "
    f"{change_rate:.2f}%"
)

print(
    f"Mean Top-1 Confidence Drop: "
    f"{mean_conf_drop:.2f} pp"
)

print(
    f"Top-5 Label Retention: "
    f"{mean_retention:.2f}%"
)

print(
    f"Top-5 Complete Disruption: "
    f"{complete_disruption:.2f}%"
)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_CSV
)